Исторический исследовательский ноутбук. Сохранён как запись этапов работы; выводы очищены, код не изменён. Ячейки содержат альтернативные сценарии и старые пути. Для повторения актуальных расчётов используйте `dissertation/run_analysis.py` и инструкции в корневом README.


In [ ]:
import pandas as pd
import numpy as np
import os
import glob

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# путь к файлам
directory_path = '/content/drive/My Drive/Colab Notebooks/phd/'

file_paths = glob.glob(os.path.join(directory_path, '*.xls')) + glob.glob(os.path.join(directory_path, '*.xlsx'))

In [ ]:
# Словарь для хранения DataFrame для каждого файла
dataframes = {}

# Считайте каждый файл в отдельный DataFrame и добавьте его в словарь
for file_path in file_paths:
    file_name = os.path.basename(file_path)  # Извлекаем имя файла для использования в качестве ключа

    try:
        if file_path.endswith('.xls'):
            # Чтение .xls файла с использованием движка xlrd
            df = pd.read_excel(file_path, engine='xlrd')
        elif file_path.endswith('.xlsx'):
            # Чтение .xlsx файла с использованием движка openpyxl
            df = pd.read_excel(file_path, engine='openpyxl')

        dataframes[file_name] = df  # Сохраняем DataFrame с именем файла

        # Печать подтверждения загрузки
        print(f"Файл {file_name} успешно загружен.")

    except Exception as e:
        print(f"Ошибка при чтении файла {file_name}: {e}")

for file_name, df in dataframes.items():
    print(f"\nДанные из файла {file_name}:")
    print(df.head())

In [ ]:
def enhance(df, value_column_name='Value'):
    # Установка строки 2 как заголовки колонок и удаление строк 0 и 1
    df.columns = df.iloc[2]  # Устанавливаем строку 2 как заголовки
    df = df.drop([0, 1, 2])  # Удаляем строки 0, 1 и 2

    # Сброс индексов для удобства
    df.reset_index(drop=True, inplace=True)

    # Удаление ненужных столбцов
    df = df.drop(columns=['Indicator Name', 'Indicator Code'])

    # Преобразование данных в плоскую таблицу
    df = df.melt(id_vars=['Country Name', 'Country Code'],
                 var_name='Year',
                 value_name='RENAME')

    # Преобразование значений в столбце 'Year' в числовой формат
    df['Year'] = df['Year'].astype(int)

    # Переименование столбца 'RENAME' в указанное имя
    df = df.rename(columns={'RENAME': value_column_name})

    return df

In [ ]:
gmi_file = 'Global Militarisation Index.xlsx'
gmi = dataframes[gmi_file]

gmi = gmi[gmi.columns.drop(list(gmi.filter(regex='^Rank')))]

# Преобразование данных в плоскую таблицу
gmi = gmi.melt(id_vars=['Country', 'ISO', 'Region'],
                    var_name='Year',
                    value_name='GMI')

# Преобразование значений в столбце
gmi['Year'] = gmi['Year'].str.extract('(\d{4})')

# Преобразование значений в столбце 'Year' в числовой формат
gmi['Year'] = gmi['Year'].astype(int)

# Переименование нескольких столбцов
gmi = gmi.rename(columns={'Country': 'Country Name',
                        'ISO': 'Country Code'})

gmi.head()

In [ ]:
fert_file = 'Adolescent fertility rate (births per 1,000 women ages 15-19).xls'
fert = dataframes[fert_file]

fert = enhance(fert, 'Fertility')

fert.head()

In [ ]:
age_dep_file = 'Age dependency ratio (% of working-age population).xls'
age_dep = dataframes[age_dep_file]

age_dep = enhance(age_dep, 'Age Dependency')

age_dep.head()

In [ ]:
armed_force_file = 'Armed forces personnel (% of total labor force).xls'
armed_force = dataframes[armed_force_file]

armed_force = enhance(armed_force, 'Armed Forces Pers')

armed_force.head()

In [ ]:
corrupt_file = 'Control of Corruption Estimate.xls'
corrupt = dataframes[corrupt_file]

corrupt = enhance(corrupt, 'Corruption')

corrupt.head()

In [ ]:
gdp_gne_gns_file = 'GDP per capita, GNS, GNE.xlsx'
gdp_gne_gns = dataframes[gdp_gne_gns_file]

# Словарь для переименования стран
rename_countries = {
    'Tьrkiye': 'Turkiye',
    'Venezuela': 'Venezuela, RB',
    'The Bahamas': 'Bahamas, The',
    'Islamic Republic of Iran': 'Iran, Islamic Rep.',
    'Syria': 'Syrian Arab Republic',
    'Egypt': 'Egypt, Arab Rep.',
    'Yemen': 'Yemen, Rep.',
    'Hong Kong SAR': 'Hong Kong SAR, China',
    'Korea': 'Korea, Rep.',
    'Lao P.D.R.': 'Lao PDR',
    'Macao SAR': 'Macao SAR, China',
    'Vietnam': 'Viet Nam',
    'Republic of Congo': 'Congo, Rep.',
    'Democratic Republic of the Congo': 'Congo, Dem. Rep.',
    'The Gambia': 'Gambia, The',
    "Cфte d'Ivoire": "Cote d'Ivoire",
    'Sгo Tomй and Prнncipe': 'Sao Tome and Principe',
    'Micronesia': 'Micronesia, Fed. Sts.',
    'Russia': 'Russian Federation',
    'Czech Republic': 'Czechia'
}

# Переименование стран в исходной таблице
gdp_gne_gns['Country'] = gdp_gne_gns['Country'].replace(rename_countries)

rename_dict = {
    'Gross domestic product based on purchasing-power-parity (PPP) share of world total': 'GDP',
    'Gross national savings': 'GNS',
    'General government total expenditure': 'GNE'
}

# Удаление ненужных столбцов
gdp_gne_gns = gdp_gne_gns.drop(columns=['Subject Notes', 'Units', 'Scale', 'Country/Series-specific Notes', 'Estimates Start After', 'WEO Country Code'])

gdp_gne_gns['Subject Descriptor'] = gdp_gne_gns['Subject Descriptor'].replace(rename_dict)

# Преобразование данных в плоскую таблицу с отдельными столбцами для каждого показателя
gdp_gne_gns = gdp_gne_gns.melt(id_vars=['Country', 'Subject Descriptor'],
                    var_name='Year',
                    value_name='Value')

# Преобразование значений в столбце 'Year' в числовой формат
gdp_gne_gns['Year'] = gdp_gne_gns['Year'].astype(int)

# Преобразование столбца 'Value' в числовой формат
gdp_gne_gns['Value'] = pd.to_numeric(gdp_gne_gns['Value'], errors='coerce')

# Поворот таблицы так, чтобы для каждого показателя был отдельный столбец
gdp_gne_gns = gdp_gne_gns.pivot_table(
    index=['Country', 'Year'],
    columns='Subject Descriptor',
    values='Value'
).reset_index()

# Создание словаря из таблицы corrupt
country_code_dict = dict(zip(corrupt['Country Name'], corrupt['Country Code']))

# Добавление столбца 'Country Code' в таблицу gdp_gne_gns, используя метод .map()
gdp_gne_gns['Country Code'] = gdp_gne_gns['Country'].map(country_code_dict)

gdp_gne_gns = gdp_gne_gns.rename(columns={'Country': 'Country Name'})

# Укажите новый порядок столбцов
new_order = ['Country Name', 'Country Code', 'Year', 'GDP', 'GNS', 'GNE']

# Изменение порядка столбцов
gdp_gne_gns = gdp_gne_gns[new_order]

gdp_gne_gns.head()

In [ ]:
govern_eff_file = 'Government Effectiveness Estimate.xls'
govern_eff = dataframes[govern_eff_file]

govern_eff = enhance(govern_eff, 'Government Eff')

govern_eff.head()

In [ ]:
life_exp_file = 'Life expectancy at birth, total (years).xls'
life_exp = dataframes[life_exp_file]

life_exp = enhance(life_exp, 'Life Expectancy')

life_exp.head()

In [ ]:
mil_exp_gdp_file = 'Military expenditure (% of GDP).xls'
mil_exp_gdp = dataframes[mil_exp_gdp_file]

mil_exp_gdp = enhance(mil_exp_gdp, 'Military Expenditure GDP')

mil_exp_gdp.head()

In [ ]:
mil_exp_gov_exp_file = 'Military expenditure (% of general government expenditure).xls'
mil_exp_gov_exp = dataframes[mil_exp_gov_exp_file]

mil_exp_gov_exp = enhance(mil_exp_gov_exp, 'Military Expenditure Gov Exp')

mil_exp_gov_exp.head()

In [ ]:
pol_stab_file = 'Political Stability and Absence of ViolenceTerrorism Estimate.xls'
pol_stab = dataframes[pol_stab_file]

pol_stab = enhance(pol_stab, 'Political Stability')

pol_stab.head()

In [ ]:
parl_file = 'Proportion of seats held by women in national parliaments (%).xls'
parl = dataframes[parl_file]

parl = enhance(parl, 'PARL')

parl.head()

In [ ]:
reg_qual_file = 'Regulatory Quality Estimate.xls'
reg_qual = dataframes[reg_qual_file]

reg_qual = enhance(reg_qual, 'Regulatory Quality')

reg_qual.head()

In [ ]:
religion_file = 'Religion.xlsx'
religion = dataframes[religion_file]

religion = religion.rename(columns={'Country': 'Country Name'})

religion.head()

In [ ]:
law_file = 'Rule of Law Estimate.xls'
law = dataframes[law_file]

law = enhance(law, 'Rule of Law')

law.head()

In [ ]:
unemployment_file = 'Unemployment, total (% of total labor force) (modeled ILO estimate).xls'
unemployment = dataframes[unemployment_file]

unemployment = enhance(unemployment, 'Unemployment')

unemployment.head()

In [ ]:
account_file = 'Voice and Accountability Estimate.xls'
account = dataframes[account_file]

account = enhance(account, 'Voice and Accountability')

account.head()

In [ ]:
women_bis_file = 'Women Business and the Law Index Score (scale 1-100).xls'
women_bis = dataframes[women_bis_file]

women_bis = enhance(women_bis, 'Women Business')

women_bis.head()

In [ ]:
def merge_all_tables(gmi, tables):

    merged_df = gmi.copy()

    # Обработка каждой таблицы из списка
    for table in tables:
        # Удаление колонки 'Country Name', если она есть
        if 'Country Name' in table.columns:
            table = table.drop(columns=['Country Name'])

        # Объединение с основной таблицей по 'Country Code' и 'Year'
        merged_df = pd.merge(merged_df, table, on=['Country Code', 'Year'], how='left')

    return merged_df

In [ ]:
# Создание списка таблиц
enhanced_tables = [fert, age_dep, armed_force, corrupt, gdp_gne_gns, govern_eff, life_exp, mil_exp_gdp, mil_exp_gov_exp, pol_stab, parl, reg_qual, law, unemployment, account, women_bis]

final_df = merge_all_tables(gmi, enhanced_tables)

final_df = pd.merge(final_df, religion[['Country Code', 'Religion']],
                        on='Country Code', how='left')

columns_order = final_df.columns.tolist()  # Список текущих столбцов
columns_order.insert(columns_order.index('Region') + 1, columns_order.pop(columns_order.index('Religion')))

# Применение нового порядка столбцов
final_df = final_df[columns_order]

final_df.head()

In [ ]:
# Замена значений '..' на NaN только в столбце 'GMI' с использованием apply()
final_df['GMI'] = final_df['GMI'].apply(lambda x: np.nan if x == '..' else x)

final_df.head()

In [ ]:
# Преобразование всех числовых столбцов в числовой формат и округление
for column in final_df.columns:
    if pd.api.types.is_numeric_dtype(final_df[column]):  # Проверка, является ли столбец числовым
        final_df[column] = pd.to_numeric(final_df[column], errors='coerce').round(5)  # Преобразование и округление

final_df.head()

In [ ]:
# Проверка на дубли по всем столбцам
duplicates = final_df.duplicated().sum()
print(f'Количество дублирующихся строк: {duplicates}')

# Проверка на дубли по определенным столбцам (например, 'Country Code' и 'Year')
duplicates_specific = final_df.duplicated(subset=['Country Code', 'Year']).sum()
print(f'Количество дублирующихся строк по "Country Code" и "Year": {duplicates_specific}')

In [ ]:
# Сохранение final_df в CSV файл с разделителем ';' и десятичным разделителем ','
final_df.to_csv('final_data.csv', sep=';', decimal=',', index=False)


In [ ]:
# Загружаем твои данные
df = pd.read_csv('data/final_data/final_data.csv', sep=';', decimal=',')

# Загружаем файл с метрикой
df_female = pd.read_csv('data/raw_data/Labor force, female (% of total labor force).csv')

# Переименовываем колонку Value
df_female = df_female.rename(columns={'Value': 'Female_Labor_Force'})

# Мержим по Country Name и Year
df = df.merge(df_female[['Country Name', 'Year', 'Female_Labor_Force']],
              on=['Country Name', 'Year'],
              how='left')

# Проверим долю пропусков
print(df['Female_Labor_Force'].isnull().mean())

# Сохраняем итоговый файл
df.to_csv('raw_data/final_data.csv', sep=';', decimal=',', index=False)

In [ ]:
from google.colab import files

# Скачивание файла
files.download('final_data.csv')

In [ ]:
import pandas as pd

df = pd.read_csv('data/final_data/final_data.csv', sep=';', decimal=',')
df_hci = pd.read_csv('data/raw_data/hci_pwt.csv')

# Посмотреть, какие вообще переменные есть (один раз, по желанию)
print(df_hci['Variable code'].unique())

# Фильтруем только human capital (если там несколько переменных)
df_hci_hc = df_hci[df_hci['Variable code'] == 'hc'].copy()
# Если в файле И ТАК только hc, то df_hci_hc = df_hci.copy()


In [ ]:
# список колонок-годов
year_cols = [str(y) for y in range(1950, 2024)]

df_hci_long = df_hci_hc.melt(
    id_vars=['ISO code', 'Country', 'Variable code', 'Variable name'],
    value_vars=year_cols,
    var_name='Year',
    value_name='Human_Capital_Index'
)


In [ ]:
df_hci_long['Year'] = df_hci_long['Year'].astype(int)

# Переименуем ISO code, чтобы совпадало с твоим df
df_hci_long = df_hci_long.rename(columns={'ISO code': 'Country Code'})


In [ ]:
df_merged = df.merge(
    df_hci_long[['Country Code', 'Year', 'Human_Capital_Index']],
    on=['Country Code', 'Year'],
    how='left'
)


In [ ]:
df_merged

In [ ]:
for c in df_merged.columns:
    if ' ' in c:
        print(c)


In [ ]:
rename_cols = {
    c: c.replace(' ', '_')
    for c in df_merged.columns
    if ' ' in c
}

df_merged = df_merged.rename(columns=rename_cols)


In [ ]:
print(df_merged.columns.tolist())


In [ ]:
df_merged = df_merged.rename(columns={
    'Women_Civil_Society_Index': 'Women_Civil_Society',
    'Human_Capital_Index': 'Human_Capital'
})


In [ ]:
df_merged.head()

In [ ]:
name_col = 'Country_Name' if 'Country_Name' in df_merged.columns else 'country_name'
code_col = 'Country_Code' if 'Country_Code' in df_merged.columns else 'country_code'

name_col, code_col


In [ ]:
# сколько разных кодов у каждого названия страны
name_to_codes = (
    df_merged
    .groupby(name_col)[code_col]
    .nunique()
    .reset_index(name='n_codes')
)

# проблемные случаи: у одного названия больше одного кода
bad_names = name_to_codes[name_to_codes['n_codes'] > 1]
print(bad_names.head(20))
print('Всего стран с >1 кодом:', len(bad_names))


In [ ]:
code_to_names = (
    df_merged
    .groupby(code_col)[name_col]
    .nunique()
    .reset_index(name='n_names')
)

bad_codes = code_to_names[code_to_names['n_names'] > 1]
print(bad_codes.head(20))
print('Всего кодов с >1 названием:', len(bad_codes))


In [ ]:
df_merged_2001 = df_merged[df_merged.Year == 2001]

In [ ]:
df_merged_2001

In [ ]:
df_merged_2002 = df_merged[df_merged.Year == 2002]
df_merged_2002

In [ ]:
df_merged.to_csv('data/final_data/final_data.csv', sep=';', decimal=',', index=False)
